In [4]:
import pandas as pd

df = pd.read_csv('../data/funnel_marketing_data.csv')
df.shape

(3500, 19)

In [5]:
# רק לקוחות שבאמת רכשו — יש להם ltv_months תקין
df_ltv = df[df['purchased'] == 1].copy()

features = ['ad_budget', 'num_leads', 'leads_answered', 'leads_not_answered',
            'followup_1', 'followup_2', 'followup_3', 'followup_4', 'followup_5',
            'calls_to_closed', 'calls_to_not_closed', 'customer_acquisition_cost']

X = df_ltv[features]
y = df_ltv['ltv_months']

X.shape, y.shape

((3163, 12), (3163,))

In [ ]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import cross_val_score, KFold

models = {
    'XGBoost': xgb.XGBRegressor(random_state=42),
    'LightGBM': lgb.LGBMRegressor(random_state=42, verbose=-1),
    'CatBoost': cb.CatBoostRegressor(random_state=42, verbose=0)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, model in models.items():
    rmse_scores = -cross_val_score(
        model, X, y, cv=kf, scoring='neg_root_mean_squared_error'
    )
    r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
    results[name] = {'RMSE': rmse_scores.mean(), 'R2': r2_scores.mean()}

results

In [8]:
xgb_model = models['XGBoost'].fit(X, y)
lgb_model = models['LightGBM'].fit(X, y)
cb_model = models['CatBoost'].fit(X, y)

importance_df = pd.DataFrame({
    'feature': features,
    'XGBoost': xgb_model.feature_importances_,
    'LightGBM': lgb_model.feature_importances_,
    'CatBoost': cb_model.feature_importances_
})

importance_df

,feature,XGBoost,LightGBM,CatBoost
0,ad_budget,0.001582,107,0.983910
1,num_leads,0.001779,437,1.669266
2,leads_answered,0.001920,318,1.092490
3,leads_not_answered,0.002968,453,1.783325
4,followup_1,0.001919,271,0.882522
5,followup_2,0.002340,210,1.244218
6,followup_3,0.002554,206,0.902962
7,followup_4,0.001880,241,0.856511
8,followup_5,0.002871,155,0.957150
9,calls_to_closed,0.974748,205,87.302807


In [ ]:
features_clean = ['ad_budget', 'num_leads', 'leads_answered', 'leads_not_answered',
                   'followup_1', 'followup_2', 'followup_3', 'followup_4', 'followup_5',
                   'customer_acquisition_cost']

X_clean = df_ltv[features_clean]

results_clean = {}
for name, model in models.items():
    rmse_scores = -cross_val_score(
        model, X_clean, y, cv=kf, scoring='neg_root_mean_squared_error'
    )
    r2_scores = cross_val_score(model, X_clean, y, cv=kf, scoring='r2')
    results_clean[name] = {'RMSE': rmse_scores.mean(), 'R2': r2_scores.mean()}

results_clean

In [10]:
xgb_model = models['XGBoost'].fit(X_clean, y)
lgb_model = models['LightGBM'].fit(X_clean, y)
cb_model = models['CatBoost'].fit(X_clean, y)

importance_df_clean = pd.DataFrame({
    'feature': features_clean,
    'XGBoost': xgb_model.feature_importances_,
    'LightGBM': lgb_model.feature_importances_,
    'CatBoost': cb_model.feature_importances_
})

importance_df_clean.sort_values('CatBoost', ascending=False)

,feature,XGBoost,LightGBM,CatBoost
0,ad_budget,0.937607,172,85.325354
9,customer_acquisition_cost,0.007147,220,4.244031
1,num_leads,0.006593,579,2.623091
5,followup_2,0.006759,264,1.393330
3,leads_not_answered,0.007974,467,1.392265
4,followup_1,0.007014,320,1.174979
2,leads_answered,0.006131,369,1.068865
6,followup_3,0.006772,210,1.045336
7,followup_4,0.007007,188,0.876155
8,followup_5,0.006995,211,0.856595
